# Notebook 2: Đánh Giá Kết Quả và Trực Quan Hóa Đối Chứng (AL vs. Random)

Notebook này đọc dữ liệu nhật ký thực nghiệm (`logs_al.json` và `logs_random.json`) đã lưu trên Google Drive để vẽ các đường cong học tập, đo lường các chỉ số tiết kiệm SSR/ESR và thực hiện kiểm định thống kê.

In [ ]:
# Cell 1: Mount Google Drive & Import Thư viện
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
# Cell 2: Tải Logs kết quả thực nghiệm từ Drive
logs_dir = "/content/drive/MyDrive/active-learning-VietBioNER/logs"

with open(os.path.join(logs_dir, "logs_al.json"), 'r', encoding='utf-8') as f:
    logs_al = json.load(f)
    
with open(os.path.join(logs_dir, "logs_random.json"), 'r', encoding='utf-8') as f:
    logs_random = json.load(f)
    
print("Đã tải log thành công!")
print(f"Nhánh AL (Đề xuất): {len(logs_al)} vòng chạy ghi nhận.")
print(f"Nhánh Random (Đối chứng): {len(logs_random)} vòng chạy ghi nhận.")

In [ ]:
# Cell 3: Vẽ biểu đồ Learning Curves (Đồ thị Hiệu năng & Chi phí)
nt_al = [step["Nt"] for step in logs_al]
f1_al = [step["f1"] * 100 for step in logs_al] # Chuyển sang %
et_al = [step["Et"] for step in logs_al]

nt_rand = [step["Nt"] for step in logs_random]
f1_rand = [step["f1"] * 100 for step in logs_random] # Chuyển sang %
et_rand = [step["Et"] for step in logs_random]

sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Đồ thị F1-score theo số lượng câu Nt
ax1.plot(nt_al, f1_al, marker='o', color='green', linewidth=2, label='Nhánh A (Active Learning + DS)')
ax1.plot(nt_rand, f1_rand, marker='s', color='grey', linestyle='--', linewidth=2, label='Nhánh B (Random + DS)')
ax1.set_title('Đường cong Học tập (F1-score theo số lượng câu gán nhãn)', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Số lượng câu được gán nhãn (Nt)', fontsize=11)
ax1.set_ylabel('F1-score (%)', fontsize=11)
ax1.legend(fontsize=10)

# 2. Đồ thị F1-score theo chi phí Levenshtein tích lũy Et
ax2.plot(et_al, f1_al, marker='o', color='green', linewidth=2, label='Nhánh A (Active Learning + DS)')
ax2.plot(et_rand, f1_rand, marker='s', color='grey', linestyle='--', linewidth=2, label='Nhánh B (Random + DS)')
ax2.set_title('Đường cong Chi phí (F1-score theo nỗ lực hiệu chỉnh Levenshtein)', fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel('Khoảng cách Levenshtein tích lũy (Et)', fontsize=11)
ax2.set_ylabel('F1-score (%)', fontsize=11)
ax2.legend(fontsize=10)

plt.tight_layout()
chart_path = os.path.join(logs_dir, "learning_curves.png")
plt.savefig(chart_path, dpi=300)
print(f"Đã lưu biểu đồ đối chứng tại: {chart_path}")
plt.show()

In [ ]:
# Cell 4: Định lượng Tỷ lệ Tiết kiệm Chi phí (SSR & ESR)
target_f1 = 75.0 # Mốc hiệu năng mục tiêu (%)

def find_metrics_at_target(nt_list, et_list, f1_list, target):
    for i, f1 in enumerate(f1_list):
        if f1 >= target:
            return nt_list[i], et_list[i]
    return nt_list[-1], et_list[-1] # Trả về điểm cuối nếu không đạt

nt_al_t, et_al_t = find_metrics_at_target(nt_al, et_al, f1_al, target_f1)
nt_rand_t, et_rand_t = find_metrics_at_target(nt_rand, et_rand, f1_rand, target_f1)

ssr = (1 - (nt_al_t / nt_rand_t)) * 100
esr = (1 - (et_al_t / et_rand_t)) * 100

print("=== BÁO CÁO ĐỊNH LƯỢNG MỨC ĐỘ TIẾT KIỆM CHI PHÍ ===")
print(f"Mốc F1-score mục tiêu xác định: {target_f1}%")
print(f"  - Nhánh A (AL) đạt được tại: Nt = {nt_al_t} câu, Et = {et_al_t} thao tác")
print(f"  - Nhánh B (Random) đạt được tại: Nt = {nt_rand_t} câu, Et = {et_rand_t} thao tác")
print("--------------------------------------------------")
print(f"Chỉ số SSR (Tỷ lệ tiết kiệm số câu): {ssr:.2f}%")
print(f"Chỉ số ESR (Tỷ lệ tiết kiệm thao tác hiệu chỉnh): {esr:.2f}%")

In [ ]:
# Cell 5: Kiểm định Ý nghĩa Thống kê (Wilcoxon Signed-Rank Test)
# Giả thuyết không H0: Hiệu năng hai nhánh tương đương nhau.
# Giả thuyết đối H1: Hiệu năng của Nhánh A (AL) vượt trội hơn Nhánh B (Random).

stat, p_value = stats.wilcoxon(f1_al, f1_rand, alternative='greater')

print("=== KẾT QUẢ KIỂM ĐỊNH THỐNG KÊ WILCOXON SIGNED-RANK TEST ===")
print(f"Wilcoxon W-statistic: {stat}")
print(f"p-value: {p_value:.6f}")
if p_value < 0.05:
    print("=> Kết quả bác bỏ giả thuyết H0 với p < 0.05. Sự vượt trội về hiệu năng học của Active Learning có ý nghĩa thống kê.")
else:
    print("=> p-value >= 0.05. Chưa đủ cơ sở thống kê bác bỏ H0. Hiệu năng chọn mẫu hai nhánh chưa có sự khác biệt rõ rệt.")